# Experiment 1: Prompt Selection

## Research Question
Which zero-shot prompt formulation yields the best intertextuality detection performance?

## Fixed Parameters
- chunk_size = 120
- overlap = 20
- num_few_shot_examples = 0
- model = gpt-5.4-mini

## Variable Parameter
- prompt_type ∈ {zero_shot_minimal, zero_shot_medium, zero_shot_maximum}

### Evaluation Procedure

The experiment is conducted using **5 evaluation folds**.  
For each fold:

1. The pipeline processes the query and source documents.
2. Results are stored to disk.
3. Predictions are converted into segment-level matches.
4. The predictions are evaluated against the ground truth.

### Metrics
- Precision
- Recall
- F1-score

Macro and micro averages are reported across folds.

In [1]:
import sys
project_src = r"C:\Users\sophi\Desktop\Master\CIS_5.Semester\Thesis\locisimiles\src"

# lokalen Pfad *vor* site-packages setzen
sys.path.insert(0, project_src)

import locisimiles
print("Jetzt geladen aus:", locisimiles.__file__)

#imports
import openai
from unittest.mock import patch
import locisimiles as ls
from locisimiles.evaluator import IntertextEvaluator
#from locisimiles.llm_pipeline_new import LLMOnlyClassificationPipeline
from locisimiles.llm_pipeline_sliding_windows import LLMOnlySlidingWindowPipeline, pretty_print, save_final_results, load_final_results
from locisimiles.document import Document, TextSegment
#from locisimiles.llm_pipeline_new import pretty_print
from locisimiles.llm_pipeline_sliding_windows import convert_llm_results, fixed_sliding_windows, convert_llm_results_coverage_fair, pad_segments
import pandas as pd
import csv
from pathlib import Path
from locisimiles.evaluator import IntertextEvaluator
import random
import tempfile
import time

# load key
from dotenv import load_dotenv
load_dotenv()

Jetzt geladen aus: C:\Users\sophi\Desktop\Master\CIS_5.Semester\Thesis\locisimiles\src\locisimiles\__init__.py


c:\Users\sophi\anaconda3\envs\py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# =========================
# DATA PATH
# =========================
DATA_ROOT = Path("../Sampled_Documents/folds")

N_FOLDS = 5

In [3]:
def load_fold(
    fold_id: int,
    *,
    root: Path = DATA_ROOT,
):
    """
    Load query document, source document and ground truth for one fold.
    """

    fold_dir = root / f"fold_{fold_id}" / "evaluation"

    if not fold_dir.exists():
        raise FileNotFoundError(f"Fold {fold_id} not found at {fold_dir}")

    query_doc = Document(fold_dir / "query_document.csv")
    source_doc = Document(fold_dir / "source_document.csv")
    ground_truth_path = fold_dir / "ground_truth.csv"

    return {
        "query": query_doc,
        "source": source_doc,
        "ground_truth": ground_truth_path,
    }

In [4]:
# =========================
# FIXED PARAMETERS
# =========================

MODEL = "gpt-5.4-mini"

CHUNK_SIZE = 120 
OVERLAP = 20
NUM_FEW_SHOT = 0

# =========================
# VARIABLE/EXPERIMENT PARAMETER
# =========================
PROMPT_TYPES = [
    "zero_shot_minimal",
    "zero_shot_medium",
    "zero_shot_maximum"
]

# ==============================
# OUTPUT
# ==============================

RESULT_DIR = Path("final_results/experiment_1_prompt_selection")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# =========================
# RUN EXPERIMENT
# =========================
for prompt_type in PROMPT_TYPES:

    print("\n====================================")
    print(f"Running experiment for prompt: {prompt_type}")
    print("====================================")

    pipeline = LLMOnlySlidingWindowPipeline(
        openai_model=MODEL,
        prompt_type=prompt_type,
        chunk_size=CHUNK_SIZE,
        overlap=OVERLAP,
        num_few_shot_examples=NUM_FEW_SHOT,
    )

    for fold_id in range(N_FOLDS):

        print(f"\nRunning fold {fold_id}")

        fold = load_fold(fold_id)

        checkpoint_path = RESULT_DIR / f"checkpoint_fold_{fold_id}_{prompt_type}.json"
        
        start_time = time.perf_counter()

        results = pipeline.run(
            query=fold["query"],
            source=fold["source"],
            checkpoint_path=checkpoint_path,
            resume=True,
        )
        
        end_time = time.perf_counter()
        runtime = end_time - start_time
        print(f"Fold {fold_id} runtime: {runtime:.2f} seconds")

        out_path = RESULT_DIR / f"fold_{fold_id}_{prompt_type}.json"

        save_final_results(
            results,
            path=out_path,
            meta={
                "model": pipeline.openai_model,
                "prompt_type": prompt_type,
                "chunk_size": pipeline.chunk_size,
                "num_few_shot_examples": pipeline.num_few_shot_examples,
                "overlap": pipeline.overlap,
                "runtime_seconds": runtime,
            },
        )

        print(f"Saved results to {out_path}")


Running experiment for prompt: zero_shot_minimal

Running fold 0
Resuming from source window 10
Total source windows: 10


Processing source windows: 100%|██████████| 10/10 [00:00<?, ?it/s]

Fold 0 runtime: 3.59 seconds


Saved results to final_results\experiment_1_prompt_selection\fold_0_zero_shot_minimal.json

Running fold 1
Resuming from source window 10
Total source windows: 10


Processing source windows: 100%|██████████| 10/10 [00:00<?, ?it/s]

Fold 1 runtime: 2.74 seconds


... (33 outputs left out)

Processing source windows: 100%|██████████| 10/10 [1:17:43<00:00, 466.36s/it]


Fold 3 runtime: 4663.64 seconds
Saved results to final_results\experiment_1_prompt_selection\fold_3_zero_shot_maximum.json

Running fold 4
Total source windows: 10


Processing source windows: 100%|██████████| 10/10 [1:20:12<00:00, 481.29s/it]


Fold 4 runtime: 4812.90 seconds
Saved results to final_results\experiment_1_prompt_selection\fold_4_zero_shot_maximum.json


In [6]:
# =========================
# EVALUATION
# =========================
all_scores = []

for prompt_type in PROMPT_TYPES:

    print("\n===============================")
    print(f"Evaluating prompt: {prompt_type}")
    print("===============================")

    for fold_id in range(N_FOLDS):

        print(f"\nEvaluating fold {fold_id}")

        fold = load_fold(fold_id)

        result_file = RESULT_DIR / f"fold_{fold_id}_{prompt_type}.json"

        raw_results = load_final_results(result_file)


        source_segments = pad_segments(
            list(fold["source"].segments.values()),
            CHUNK_SIZE,
            prefix="S"
        )

        query_segments = pad_segments(
            list(fold["query"].segments.values()),
            CHUNK_SIZE,
            prefix="Q"
        )

        source_windows = fixed_sliding_windows(
            source_segments,
            CHUNK_SIZE,
            pipeline.stride
        )

        query_windows = fixed_sliding_windows(
            query_segments,
            CHUNK_SIZE,
            pipeline.stride
        )

        full_results = convert_llm_results_coverage_fair(
            raw_results,
            fold["source"],
            fold["query"],
            source_windows,
            query_windows,
            threshold=0.0
        )

        wrapped_pipeline = type(
            "DummyPipeline",
            (),
            {"run": lambda self, **kwargs: full_results}
        )()

        evaluator = IntertextEvaluator(
            query_doc=fold["query"],
            source_doc=fold["source"],
            ground_truth_csv=fold["ground_truth"],
            pipeline=wrapped_pipeline,
            top_k=1000,
            threshold=0.5,
        )

        macro_scores = evaluator.evaluate(average="macro")
        micro_scores = evaluator.evaluate(average="micro")

        print("Macro:", macro_scores)
        print("Micro:", micro_scores)
        
        macro_row = macro_scores.iloc[0]
        micro_row = micro_scores.iloc[0]

        all_scores.append({
            "prompt_type": prompt_type,
            "fold": fold_id,

            "macro_precision": macro_row["precision"],
            "macro_recall": macro_row["recall"],
            "macro_f1": macro_row["f1"],
            "macro_accuracy": macro_row["accuracy"],
            "macro_fpr": macro_row["fpr"],
            "macro_fnr": macro_row["fnr"],
            "macro_smr": macro_row["smr"],
            "macro_tp": macro_row["tp"],
            "macro_fp": macro_row["fp"],
            "macro_fn": macro_row["fn"],
            "macro_tn": macro_row["tn"],

            "micro_precision": micro_row["precision"],
            "micro_recall": micro_row["recall"],
            "micro_f1": micro_row["f1"],
            "micro_accuracy": micro_row["accuracy"],
            "micro_fpr": micro_row["fpr"],
            "micro_fnr": micro_row["fnr"],
            "micro_smr": micro_row["smr"],
            "micro_tp": micro_row["tp"],
            "micro_fp": micro_row["fp"],
            "micro_fn": micro_row["fn"],
            "micro_tn": micro_row["tn"],
        })


Evaluating prompt: zero_shot_minimal

Evaluating fold 0

PAIR: <hier. epist. 66.3.1.2> || <cic. fin. 5.67.2>
  labels: [1]
  sum(labels): 1
  len(labels): 1
  coverage_source: 1
  coverage_query: 1
  max_labels: 1
  prob: 1.0000
  observed/max: 1.0000
--------------------------------------------------

PAIR: <hier. epist. 119.11.1.2> || <cic. nat. deor. 1.10.3>
  labels: [0, 1]
  sum(labels): 1
  len(labels): 2
  coverage_source: 1
  coverage_query: 2
  max_labels: 2
  prob: 0.5000
  observed/max: 1.0000
--------------------------------------------------

PAIR: <hier. epist. 35.1.3.3> || <cic. de orat. 1.32.3>
  labels: [1]
  sum(labels): 1
  len(labels): 1
  coverage_source: 1
  coverage_query: 1
  max_labels: 1
  prob: 1.0000
  observed/max: 1.0000
--------------------------------------------------

PAIR: <hier. epist. 35.1.3.3> || <cic. de orat. 1.33.1>
  labels: [1]
  sum(labels): 1
  len(labels): 1
  coverage_source: 1
  coverage_query: 1
  max_labels: 1
  prob: 1.0000
  observed

In [10]:
# =========================
# RESULTS AGGREGATION
# =========================
df = pd.DataFrame(all_scores)

summary = (
    df.groupby("prompt_type")
    .mean(numeric_only=True)
    .sort_values("macro_f1", ascending=False)
)

summary

,fold,macro_precision,macro_recall,macro_f1,macro_accuracy,macro_fpr,macro_fnr,macro_smr,macro_tp,macro_fp,...,micro_recall,micro_f1,micro_accuracy,micro_fpr,micro_fnr,micro_smr,micro_tp,micro_fp,micro_fn,micro_tn
prompt_type,,,,,,,,,,,,,,,,,,,,,
zero_shot_maximum,2.0,0.066541,0.075262,0.069016,0.999906,0.000050,0.000043,0.000094,73.0,41.4,...,0.670931,0.654612,0.999906,0.000050,0.000043,0.000094,73.0,41.4,35.8,823001.8
zero_shot_medium,2.0,0.064418,0.073904,0.066940,0.999886,0.000069,0.000045,0.000114,72.0,57.2,...,0.661961,0.601572,0.999886,0.000069,0.000045,0.000114,72.0,57.2,36.8,822986.0
zero_shot_minimal,2.0,0.057226,0.064248,0.059329,0.999911,0.000033,0.000056,0.000089,62.6,27.2,...,0.575416,0.628496,0.999911,0.000033,0.000056,0.000089,62.6,27.2,46.2,823016.0


In [9]:
df

,prompt_type,fold,macro_precision,macro_recall,macro_f1,macro_accuracy,macro_fpr,macro_fnr,macro_smr,macro_tp,...,micro_recall,micro_f1,micro_accuracy,micro_fpr,micro_fnr,micro_smr,micro_tp,micro_fp,micro_fn,micro_tn
0,zero_shot_minimal,0,0.065902,0.074707,0.068339,0.999913,0.000042,0.000045,0.000087,72.0,...,0.660550,0.666667,0.999913,0.000042,0.000045,0.000087,72.0,35.0,37.0,824416.0
1,zero_shot_minimal,1,0.049180,0.056684,0.051378,0.999903,0.000033,0.000064,0.000097,56.0,...,0.513761,0.583333,0.999903,0.000033,0.000064,0.000097,56.0,27.0,53.0,822664.0
2,zero_shot_minimal,2,0.051227,0.054963,0.052259,0.999903,0.000028,0.000069,0.000097,52.0,...,0.477064,0.565217,0.999903,0.000028,0.000069,0.000097,52.0,23.0,57.0,824428.0
3,zero_shot_minimal,3,0.059498,0.065591,0.061521,0.999924,0.000026,0.000050,0.000076,68.0,...,0.623853,0.686869,0.999924,0.000026,0.000050,0.000076,68.0,21.0,41.0,818270.0
4,zero_shot_minimal,4,0.060323,0.069296,0.063149,0.999912,0.000036,0.000052,0.000088,65.0,...,0.601852,0.640394,0.999912,0.000036,0.000052,0.000088,65.0,30.0,43.0,825302.0
5,zero_shot_medium,0,0.073372,0.084312,0.076556,0.999890,0.000076,0.000034,0.000110,81.0,...,0.743119,0.640316,0.999890,0.000076,0.000034,0.000110,81.0,63.0,28.0,824388.0
6,zero_shot_medium,1,0.054367,0.064706,0.057540,0.999878,0.000067,0.000055,0.000122,64.0,...,0.587156,0.561404,0.999878,0.000067,0.000055,0.000122,64.0,55.0,45.0,822636.0
7,zero_shot_medium,2,0.049982,0.056564,0.051725,0.999876,0.000058,0.000065,0.000124,55.0,...,0.504587,0.518868,0.999876,0.000058,0.000065,0.000124,55.0,48.0,54.0,824403.0
8,zero_shot_medium,3,0.068943,0.075986,0.069739,0.999890,0.000071,0.000039,0.000110,77.0,...,0.706422,0.631148,0.999890,0.000071,0.000039,0.000110,77.0,58.0,32.0,818233.0
9,zero_shot_medium,4,0.075426,0.087953,0.079140,0.999895,0.000075,0.000030,0.000105,83.0,...,0.768519,0.656126,0.999895,0.000075,0.000030,0.000105,83.0,62.0,25.0,825270.0


In [11]:
# =========================
# SAVE RESULT EVALUATION
# =========================
summary_path = RESULT_DIR / "prompt_experiment_summary.csv"

summary.to_csv(summary_path, index=False)
df.to_csv(RESULT_DIR / "prompt_experiment_results.csv", index=False)

print("Saved summary to", summary_path)

Saved summary to final_results\experiment_1_prompt_selection\prompt_experiment_summary.csv
